In [1]:
import geopandas as gpd
import rasterio
import geemap
import torch
import pyproj
print('geopandas:', gpd.__version__)
print('pyproj:', pyproj.__version__)
print('torch:', torch.__version__)
print('MPS available:', torch.backends.mps.is_available())
print('All good')

geopandas: 1.1.3
pyproj: 3.7.2
torch: 2.10.0
MPS available: True
All good


In [2]:
#Importing necessary libraries
import ee
import geemap

In [3]:
import ee
print(ee.__version__)

1.7.26


In [4]:
ee.Authenticate(auth_mode='notebook')

True

In [5]:
ee.Initialize(project='nowmi-learn')

In [6]:
print(ee.String("Hello from GEE").getInfo())

Hello from GEE


In [7]:
import geopandas as gpd

In [8]:
gdf = gpd.read_file('Administrative_boundaries_extract_for_Chennai_regional_level__-6243493667971233304/Administrative_boundaries_extract_for_Chennai_regional_level__0.shp')

In [9]:
roi = geemap.geopandas_to_ee(gdf)
roi_geometry = roi.geometry()
roi

In [10]:
roi_geometry

ee.Geometry({
  "functionInvocationValue": {
    "functionName": "Collection.geometry",
    "arguments": {
      "collection": {
        "functionInvocationValue": {
          "functionName": "Collection",
          "arguments": {
            "features": {
              "arrayValue": {
                "values": [
                  {
                    "functionInvocationValue": {
                      "functionName": "Feature",
                      "arguments": {
                        "geometry": {
                          "functionInvocationValue": {
                            "functionName": "GeometryConstructors.MultiPolygon",
                            "arguments": {
                              "coordinates": {
                                "constantValue": [
                                  [
                                    [
                                      [
                                        80.3181441729928,
                                        13.437679392863703
                                      ],
                                      [
                                        80.31702939246449,
                                        13.438981454644097
                                      ],
                                      [
                                        80.31651662587379,
                                        13.440154150039298
                                      ],
                                      [
                                        80.31787665761318,
                                        13.442646793560597
                                      ],
                                      [
                                        80.31844739006569,
                                        13.442896568511296
                                      ],
                                      [
                                        80.31958450812598,
                                        13.443190860987603
                                      ],
                                      [
                                        80.32007940513128,
                                        13.44275381783481
                                      ],
                                      [
                                        80.3204093915227,
                                        13.440051589157594
                                      ],
                                      [
                                        80.32018200536619,
                                        13.438218962713092
                                      ],
                                      [
                                        80.3181441729928,
                                        13.437679392863703
                                      ]
                                    ]
                                  ],
                                  [
                                    [
                                      [
                                        80.318416192845,
                                        13.4134352791365
                                      ],
                                      [
                                        80.3160529060538,
                                        13.413769712232092
                                      ],
                                      [
                                        80.31368504742929,
                                        13.414991489441295
                                      ],
                                      [
                                        80.31264612357978,
                                        13.421755974786096
                                      ],
                                      [
                                        80.313435370229,
                                        13.424056798011694
                                      ],
                   

In [11]:
Map = geemap.Map()

In [12]:
Map.centerObject(roi_geometry)

In [13]:
Map.addLayer(roi_geometry)
Map

Map(center=[12.895956588028298, 79.81602936183398], controls=(WidgetControl(options=['position', 'transparent_…

In [16]:
before_c = (ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(roi_geometry)
    .filterDate('2023-11-29','2023-12-01')
    .filter(ee.Filter.eq('instrumentMode','IW'))
    .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
    .filter(ee.Filter.eq('resolution_meters',10))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation','VH'))
    .select('VH')).mosaic().clip(roi_geometry)

In [17]:
before_c

In [18]:
after_c = (ee.ImageCollection('COPERNICUS/S1_GRD')
         .filterBounds(roi_geometry)
         .filterDate('2024-01-16','2024-01-18')
         .filter(ee.Filter.eq('instrumentMode','IW'))
         .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
         .filter(ee.Filter.eq('resolution_meters',10))
         .filter(ee.Filter.listContains('transmitterReceiverPolarisation','VH'))
         .select('VH')).mosaic().clip(roi_geometry)

In [19]:
after_c

In [20]:
Map.addLayer(before_c, {'min': -25, 'max': 0}, 'Before Flood')
Map.addLayer(after_c, {'min': -25, 'max': 0}, 'After Flood')

In [12]:
Map.addLayerControl()

In [21]:
Map

Map(bottom=61077.0, center=[12.959306528626138, 79.86284170286694], controls=(WidgetControl(options=['position…

In [22]:
# Speckle filtering - simple method is by applying a focal median of 30 with circle filter. 
# Right scientific approach is by using Refined Lee filter
smoothing_radius = 1
before_filtered = before_c.focal_median(smoothing_radius, 'square')
after_filtered = after_c.focal_median(smoothing_radius, 'square')


In [23]:
Map1 = geemap.Map()

In [24]:
Map1.addLayer(before_c, {'min': -25, 'max': 0}, 'Before Flood')
Map1.addLayer(after_c, {'min': -25, 'max': 0}, 'After Flood')
Map1.addLayer(before_filtered, {'min': -25, 'max': 0}, 'Before Flood Filtered')
Map1.addLayer(after_filtered, {'min': -25, 'max': 0}, 'After Flood Filtered')

In [25]:
Map1.centerObject(roi_geometry)

In [26]:
Map1

Map(center=[12.895956588028298, 79.81602936183398], controls=(WidgetControl(options=['position', 'transparent_…

In [32]:
difference = after_filtered.subtract(before_filtered)
# since GRD is log scale subtraction makes sense...!?
threshold = -5
flooded = difference.lt(threshold).rename('water').selfMask()
Map1.addLayer(flooded, {'palette': ['blue']}, 'Initial Flood Area')

In [35]:
gsw = ee.Image("JRC/GSW1_4/GlobalSurfaceWater")

In [36]:
gsw

In [37]:
permanent = gsw.select('occurrence').gt(90)

In [38]:
permanent

In [39]:
permanent.selfMask()

In [40]:
Map1.addLayer(permanent)